# 11.01 -- GPU Memory Planning and Model Placement

**Module:** 11 -- LLM Systems Engineering  
**Notebook:** 1 of 5  
**Prerequisites:** 09.01 (KV Cache), 09.04 (Quantisation)

---

## Learning Objectives

1. Build a precise GPU memory budget for any LLM: weights, KV cache, activations, and framework overhead
2. Understand the memory hierarchy: HBM, L2 cache, shared memory, and registers
3. Derive the maximum batch size and context length subject to a memory constraint
4. Implement a model placement calculator for multi-GPU setups
5. Measure real memory consumption and compare against theoretical predictions

---

## 1. The GPU Memory Budget

Every LLM deployment starts with the same question: does this model fit on my hardware?
Getting the answer wrong in either direction is expensive: underestimating causes OOM crashes
at runtime; overestimating means buying more GPUs than necessary.

GPU memory is consumed by four distinct components:

```
Total GPU Memory = Model Weights
                 + KV Cache
                 + Activations (during forward pass)
                 + Framework Overhead (CUDA context, PyTorch allocator, etc.)
```

Each component has a precise formula, and understanding all four allows you to plan
deployments, set `gpu_memory_utilization` in vLLM, and avoid surprise OOM errors.

---

## 2. Memory Components

**Model weights**: the dominant term for large models.
```
  weight_bytes = n_params * bytes_per_param
  bytes_per_param: FP32=4, FP16/BF16=2, INT8=1, INT4=0.5
```

**KV cache**: grows with batch size and sequence length.
```
  kv_bytes = 2 * n_layers * n_kv_heads * head_dim * seq_len * batch_size * bytes_per_element
  Factor 2: keys AND values
  head_dim = hidden_size / n_heads (= 128 for most modern models)
```

**Activations**: temporary tensors during the forward pass.
```
  activation_bytes ~ batch_size * seq_len * hidden_size * n_layers * 2 * bytes
  (rough estimate; varies by architecture and chunked prefill settings)
```

**Framework overhead**: ~1-2 GB fixed cost for CUDA context, PyTorch allocator,
NCCL buffers (multi-GPU), and temporary computation buffers.


In [ ]:
# Environment setup.
#
# This notebook is primarily analytical: we build precise memory models and
# validate them against real PyTorch measurements. The model architecture
# parameters (hidden_size, n_layers, etc.) are the inputs; memory bytes are
# the outputs. We use GPT-2 and LLaMA-style configs for concrete examples.

import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional
import math
import warnings
warnings.filterwarnings('ignore')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
HAS_CUDA = device == 'cuda'
print(f'Device: {device}')
if HAS_CUDA:
    props = torch.cuda.get_device_properties(0)
    print(f'GPU: {props.name}')
    print(f'  Total memory:      {props.total_memory / 1e9:.1f} GB')
    print(f'  Memory bandwidth:  ~{props.memory_clock_rate * props.memory_bus_width / 8e6:.0f} GB/s (theoretical)')

def gb(n_bytes): return n_bytes / 1024**3
def mb(n_bytes): return n_bytes / 1024**2


## 3. Memory Budget Calculator


In [ ]:
# LLM memory budget calculator.
#
# ModelConfig holds the architecture hyperparameters needed to compute memory.
# These are available from any model config.json in the HuggingFace hub.
#
# MemoryBudget breaks down consumption into the four components so engineers
# can understand which term dominates and which levers to pull:
#
#   weights:     reduce with quantisation (INT4/INT8) or a smaller model
#   kv_cache:    reduce with GQA/MQA, smaller context window, or KV quantisation
#   activations: reduce with chunked prefill (lower peak) or gradient checkpointing
#   overhead:    fixed; budget 1.5 GB as a conservative estimate

@dataclass
class ModelConfig:
    name:          str
    n_params:      int     # total parameter count
    n_layers:      int     # transformer decoder layers
    hidden_size:   int     # model dimension
    n_heads:       int     # attention heads
    n_kv_heads:    int     # key-value heads (= n_heads for MHA; < n_heads for GQA)
    vocab_size:    int
    max_seq_len:   int     # maximum supported context window

    @property
    def head_dim(self): return self.hidden_size // self.n_heads


@dataclass
class MemoryBudget:
    weights_gb:     float
    kv_cache_gb:    float
    activations_gb: float
    overhead_gb:    float = 1.5

    @property
    def total_gb(self): return self.weights_gb + self.kv_cache_gb + self.activations_gb + self.overhead_gb

    def fits_on(self, gpu_gb: float) -> bool: return self.total_gb <= gpu_gb

    def summary(self, model_name: str = '') -> str:
        label = f'  [{model_name}]' if model_name else ''
        return (
            f'{label}\n'
            f'  Weights:     {self.weights_gb:>6.2f} GB\n'
            f'  KV cache:    {self.kv_cache_gb:>6.2f} GB\n'
            f'  Activations: {self.activations_gb:>6.2f} GB\n'
            f'  Overhead:    {self.overhead_gb:>6.2f} GB\n'
            f'  TOTAL:       {self.total_gb:>6.2f} GB'
        )


def compute_memory_budget(
    config:        ModelConfig,
    batch_size:    int,
    seq_len:       int,
    dtype_bytes:   int   = 2,    # FP16 default
    kv_dtype_bytes: int  = 2,    # KV cache dtype (can quantise separately)
) -> MemoryBudget:
    """
    Compute the full GPU memory budget for a given model and serving configuration.

    All formulas are derived from first principles:
      - Weight bytes:      n_params * dtype_bytes
      - KV cache bytes:    2 * n_layers * n_kv_heads * head_dim * seq_len * batch_size * kv_dtype
      - Activation bytes:  batch_size * seq_len * hidden_size * 2 * dtype_bytes
        (factor 2 for input/output activations; rough estimate for MLP layers)
    """
    weights_bytes = config.n_params * dtype_bytes

    kv_bytes = (
        2                        # keys + values
        * config.n_layers
        * config.n_kv_heads
        * config.head_dim
        * seq_len
        * batch_size
        * kv_dtype_bytes
    )

    # Activation estimate: one activation tensor per layer, two layers (attn + MLP)
    activation_bytes = (
        batch_size * seq_len * config.hidden_size
        * 2 * config.n_layers * dtype_bytes
    )

    return MemoryBudget(
        weights_gb     = gb(weights_bytes),
        kv_cache_gb    = gb(kv_bytes),
        activations_gb = gb(activation_bytes),
    )


# Reference model configurations (from public model cards)
CONFIGS = {
    'GPT-2 (117M)':       ModelConfig('GPT-2',       117_000_000,   12,  768,  12,  12, 50257,  1024),
    'LLaMA-3 1B':         ModelConfig('LLaMA-3-1B',  1_240_000_000, 16, 2048,  32,   8, 128256, 8192),
    'LLaMA-3 8B':         ModelConfig('LLaMA-3-8B',  8_030_000_000, 32, 4096,  32,   8, 128256, 8192),
    'LLaMA-3 70B':        ModelConfig('LLaMA-3-70B', 70_600_000_000,80, 8192,  64,   8, 128256, 8192),
    'Mistral 7B':         ModelConfig('Mistral-7B',  7_250_000_000, 32, 4096,  32,   8, 32000,  8192),
    'Qwen2 72B':          ModelConfig('Qwen2-72B',   72_700_000_000,80, 8192,  64,   8, 152064,131072),
}

# Print memory budgets for a typical interactive serving scenario
BATCH = 8; SEQ = 2048
print(f'Memory budgets for batch_size={BATCH}, seq_len={SEQ}, FP16 weights, FP16 KV cache:')
print('=' * 60)
for name, cfg in CONFIGS.items():
    budget = compute_memory_budget(cfg, BATCH, SEQ)
    flag   = '' if budget.total_gb > 200 else (' fits A100-80GB' if budget.total_gb <= 80 else ' fits 2xA100')
    print(f'{name:<22}  total={budget.total_gb:>6.1f} GB  (w={budget.weights_gb:.1f} kv={budget.kv_cache_gb:.1f}){flag}')


In [ ]:
# Maximum batch size and context length solver.
#
# Given a fixed GPU memory budget, we can solve for the maximum batch size
# subject to a fixed context length, or vice versa. This is the 'memory solver'
# that tools like vLLM run during startup to determine how much KV cache to
# pre-allocate.
#
# The solver iterates over possible values, checks the memory constraint,
# and returns the maximum feasible configuration. In production systems this
# is solved analytically or with a binary search.
#
# The key insight is that KV cache is the dominant variable-cost term.
# Weights are fixed; KV cache grows linearly with batch_size * seq_len.
# So the feasible region is a hyperbola in (batch_size, seq_len) space.

def max_batch_for_seqlen(
    config:       ModelConfig,
    gpu_gb:       float,
    seq_len:      int,
    dtype_bytes:  int = 2,
    kv_bytes:     int = 2,
    overhead_gb:  float = 1.5,
) -> int:
    """
    Return the maximum batch size that fits within gpu_gb memory
    at the given sequence length.
    """
    weight_gb = gb(config.n_params * dtype_bytes)
    available = gpu_gb - weight_gb - overhead_gb
    if available <= 0:
        return 0

    # KV cache per sequence
    kv_per_seq_gb = gb(
        2 * config.n_layers * config.n_kv_heads * config.head_dim * seq_len * kv_bytes
    )
    # Activation per sequence (rough)
    act_per_seq_gb = gb(
        seq_len * config.hidden_size * 2 * config.n_layers * dtype_bytes
    )
    per_seq = kv_per_seq_gb + act_per_seq_gb
    if per_seq <= 0:
        return 0
    return max(0, int(available / per_seq))


# Effect of context length on max batch size across GPU sizes
GPU_CONFIGS = [
    ('RTX 4090 (24 GB)', 24),
    ('A100-40GB (40 GB)', 40),
    ('A100-80GB (80 GB)', 80),
    ('H100 SXM (80 GB)',  80),
]
SEQ_LENS     = [512, 1024, 2048, 4096, 8192]
TARGET_MODEL = CONFIGS['LLaMA-3 8B']

fig = plt.figure(figsize=(15, 10))
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.40, wspace=0.32)

# Panel 1: max batch size vs context length per GPU
ax = fig.add_subplot(gs[0, 0])
colors_gpu = ['#d62728', '#ff7f0e', '#2ca02c', '#1f77b4']
for (gpu_name, gpu_gb), col in zip(GPU_CONFIGS, colors_gpu):
    max_batches = [max_batch_for_seqlen(TARGET_MODEL, gpu_gb, sl) for sl in SEQ_LENS]
    ax.plot(SEQ_LENS, max_batches, 'o-', color=col, lw=2, ms=6, label=gpu_name)
ax.set_xlabel('Context length (tokens)', fontsize=11)
ax.set_ylabel('Max batch size', fontsize=11)
ax.set_title(f'Max Batch Size vs Context Length\n({TARGET_MODEL.name}, FP16)', fontsize=12)
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# Panel 2: memory breakdown by component (stacked bars, LLaMA 8B, A100-80GB)
ax = fig.add_subplot(gs[0, 1])
batch_sizes_test = [1, 4, 8, 16, 32]
w_vals, kv_vals, act_vals, oh_vals = [], [], [], []
for bs in batch_sizes_test:
    b = compute_memory_budget(TARGET_MODEL, bs, 2048)
    w_vals.append(b.weights_gb); kv_vals.append(b.kv_cache_gb)
    act_vals.append(b.activations_gb); oh_vals.append(b.overhead_gb)

x = np.arange(len(batch_sizes_test))
ax.bar(x, w_vals,   label='Weights',     color='#1f77b4', alpha=0.9)
ax.bar(x, kv_vals,  bottom=w_vals,       label='KV cache',    color='#ff7f0e', alpha=0.9)
act_bottom = [a+b for a,b in zip(w_vals, kv_vals)]
ax.bar(x, act_vals, bottom=act_bottom,   label='Activations', color='#2ca02c', alpha=0.9)
oh_bottom  = [a+b for a,b in zip(act_bottom, act_vals)]
ax.bar(x, oh_vals,  bottom=oh_bottom,    label='Overhead',    color='#d62728', alpha=0.9)
ax.axhline(80, color='black', ls='--', lw=1.5, label='A100-80GB limit')
ax.set_xticks(x)
ax.set_xticklabels([f'bs={b}' for b in batch_sizes_test])
ax.set_ylabel('GPU Memory (GB)', fontsize=11)
ax.set_title(f'Memory Breakdown by Component\n({TARGET_MODEL.name}, seq=2048, FP16)', fontsize=12)
ax.legend(fontsize=8, loc='upper left')
ax.grid(alpha=0.3, axis='y')

# Panel 3: effect of quantisation on required GPUs
ax = fig.add_subplot(gs[1, 0])
quant_configs = [
    ('FP32',  4, 4), ('FP16',  2, 2), ('BF16',  2, 2),
    ('INT8',  1, 2), ('NF4/INT4', 0.5, 2),
]
for model_name in ['LLaMA-3 8B', 'LLaMA-3 70B']:
    cfg = CONFIGS[model_name]
    total_mems = []
    for qlabel, w_bpp, kv_bpp in quant_configs:
        b = compute_memory_budget(cfg, 8, 2048, dtype_bytes=w_bpp, kv_dtype_bytes=kv_bpp)
        total_mems.append(b.total_gb)
    ax.plot([q[0] for q in quant_configs], total_mems, 'o-', lw=2, ms=7, label=model_name)
for gpu_thresh, gpu_lbl in [(24,'24 GB'), (40,'40 GB'), (80,'80 GB')]:
    ax.axhline(gpu_thresh, ls=':', alpha=0.5, color='gray')
    ax.text(len(quant_configs)-0.8, gpu_thresh+0.5, gpu_lbl, fontsize=8, color='gray')
ax.set_xlabel('Quantisation', fontsize=11)
ax.set_ylabel('Total GPU memory (GB)', fontsize=11)
ax.set_title('Quantisation vs Required GPU Memory\n(batch=8, seq=2048)', fontsize=12)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# Panel 4: KV cache scaling with GQA
ax = fig.add_subplot(gs[1, 1])
# Compare MHA (n_kv_heads = n_heads) vs GQA (n_kv_heads = 8) vs MQA (n_kv_heads = 1)
seq_lens_kv = list(range(512, 16385, 512))
for kv_heads, label, col in [
    (32, 'MHA (n_kv=32)', '#d62728'),
    (8,  'GQA (n_kv=8)',  '#2ca02c'),
    (1,  'MQA (n_kv=1)',  '#1f77b4'),
]:
    kv_cfg = ModelConfig('test', TARGET_MODEL.n_params, TARGET_MODEL.n_layers,
                         TARGET_MODEL.hidden_size, TARGET_MODEL.n_heads,
                         kv_heads, TARGET_MODEL.vocab_size, TARGET_MODEL.max_seq_len)
    kv_mems = [gb(2 * kv_cfg.n_layers * kv_heads * kv_cfg.head_dim * sl * 8 * 2)
               for sl in seq_lens_kv]
    ax.plot(seq_lens_kv, kv_mems, lw=2, color=col, label=label)
ax.set_xlabel('Sequence length (tokens)', fontsize=11)
ax.set_ylabel('KV cache memory (GB)', fontsize=11)
ax.set_title('KV Cache Memory: MHA vs GQA vs MQA\n(batch=8, FP16, 8B model)', fontsize=12)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

plt.suptitle('GPU Memory Planning for LLM Deployment', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/11_memory_planning.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Measure real GPU memory usage vs theoretical prediction.
#
# We load GPT-2 in FP16 and measure memory at each stage:
#   Stage 0 (baseline): CUDA context only
#   Stage 1 (weights):  after loading model weights
#   Stage 2 (forward):  peak during a forward pass with batch_size=4, seq_len=128
#
# The difference between stages tells us exactly what each component costs.
# Comparing against the theoretical prediction validates our formula.
#
# We use torch.cuda.memory_allocated() which reports bytes currently in use
# by PyTorch tensors, and torch.cuda.max_memory_allocated() for the peak.
# Note: memory_allocated < memory_reserved (PyTorch pre-reserves blocks).

from transformers import AutoModelForCausalLM, AutoTokenizer
import gc

if HAS_CUDA:
    torch.cuda.empty_cache(); gc.collect()
    mem_baseline = torch.cuda.memory_allocated() / 1024**3

    tokenizer = AutoTokenizer.from_pretrained('gpt2')
    tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained('gpt2', torch_dtype=torch.float16).to('cuda')
    model.eval()
    mem_after_weights = torch.cuda.memory_allocated() / 1024**3

    torch.cuda.reset_peak_memory_stats()
    ids = tokenizer(['The attention mechanism'] * 4, return_tensors='pt',
                    padding=True).input_ids.to('cuda')
    with torch.no_grad():
        model.generate(ids, max_new_tokens=32, pad_token_id=tokenizer.eos_token_id)
    mem_peak_forward = torch.cuda.max_memory_allocated() / 1024**3

    cfg_gpt2 = CONFIGS['GPT-2 (117M)']
    theory   = compute_memory_budget(cfg_gpt2, batch_size=4, seq_len=128+32,
                                     dtype_bytes=2, kv_dtype_bytes=2)

    print('Real vs theoretical memory for GPT-2 (FP16, batch=4, seq~160):')
    print(f'  Baseline (CUDA context):    {mem_baseline:.3f} GB')
    print(f'  After weights:              {mem_after_weights:.3f} GB'
          f'  (theory: {theory.weights_gb:.3f} GB)')
    print(f'  Peak during generation:     {mem_peak_forward:.3f} GB'
          f'  (theory total: {theory.total_gb:.3f} GB)')
    print(f'  Weight measurement error:   {abs(mem_after_weights - theory.weights_gb):.3f} GB')
else:
    print('GPU not available; showing theoretical prediction only.')
    cfg_gpt2 = CONFIGS['GPT-2 (117M)']
    theory = compute_memory_budget(cfg_gpt2, 4, 160)
    print(theory.summary('GPT-2 (117M), bs=4, seq=160, FP16'))


In [ ]:
# Multi-GPU model placement calculator.
#
# When a model does not fit on a single GPU, we have two strategies:
#
#   Tensor Parallelism (TP): split each weight matrix across N GPUs.
#     Each GPU holds 1/N of each layer. All GPUs run every layer in parallel.
#     Requires all-reduce communication at each layer boundary.
#     Best for: maximising throughput, latency-sensitive workloads.
#
#   Pipeline Parallelism (PP): assign consecutive layers to consecutive GPUs.
#     GPU 0 runs layers 0..K, GPU 1 runs layers K+1..2K, etc.
#     Requires point-to-point communication between pipeline stages.
#     Best for: large batch throughput, can overlap communication with compute.
#
# The placement calculator tells you the minimum GPU count for each strategy.

def min_gpus_needed(
    config:      ModelConfig,
    gpu_gb:      float,
    batch_size:  int   = 8,
    seq_len:     int   = 2048,
    dtype_bytes: int   = 2,
) -> Dict:
    """
    Compute the minimum number of GPUs required to serve this model.

    Tensor parallelism: each GPU holds 1/N weights + 1/N KV cache.
    Pipeline parallelism: each GPU holds 1/N layers, full KV for those layers.

    Returns a dict with TP and PP minimums and memory per GPU.
    """
    full_budget = compute_memory_budget(config, batch_size, seq_len, dtype_bytes)

    # Tensor parallelism: memory scales as 1/N for both weights and KV cache
    # Communication overhead: ~5-10% extra for NCCL all-reduces
    tp_overhead = 1.10
    tp_n = 1
    while (full_budget.total_gb * tp_overhead) / tp_n + 1.5 > gpu_gb:
        tp_n *= 2
        if tp_n > 64: break

    # Pipeline parallelism: weights split, KV cache replicated per GPU
    # Each GPU has: 1/N weights + full KV cache for its layers
    pp_n = 1
    while True:
        weights_per_gpu = full_budget.weights_gb / pp_n
        kv_per_gpu      = full_budget.kv_cache_gb / pp_n   # KV also split by layers
        total_per_gpu   = weights_per_gpu + kv_per_gpu + full_budget.overhead_gb
        if total_per_gpu <= gpu_gb:
            break
        pp_n += 1
        if pp_n > 64: break

    return {
        'model':         config.name,
        'full_memory_gb': full_budget.total_gb,
        'tp_min_gpus':   tp_n,
        'tp_mem_per_gpu': full_budget.total_gb / tp_n,
        'pp_min_gpus':   pp_n,
        'pp_mem_per_gpu': full_budget.weights_gb / pp_n + full_budget.kv_cache_gb / pp_n + 1.5,
    }


print('Model placement calculator (A100-80GB, batch=8, seq=2048, FP16):')
print(f'{"Model":<22} {"Full GB":>8} {"TP GPUs":>8} {"PP GPUs":>8}')
print('-' * 52)
for model_name, cfg in CONFIGS.items():
    r = min_gpus_needed(cfg, gpu_gb=80, batch_size=8, seq_len=2048)
    print(f'{model_name:<22} {r["full_memory_gb"]:>8.1f} {r["tp_min_gpus"]:>8} {r["pp_min_gpus"]:>8}')


## 4. Engineering Notes

**Memory planning checklist before deployment**

| Step | Action |
|------|--------|
| 1 | Compute theoretical weight memory: `n_params * dtype_bytes` |
| 2 | Estimate KV cache at target batch_size and seq_len |
| 3 | Add 1.5 GB fixed overhead |
| 4 | Add 20% safety margin for allocator fragmentation |
| 5 | Set vLLM `gpu_memory_utilization = (model_gb) / (total_gpu_gb)` |
| 6 | Monitor `torch.cuda.memory_allocated()` in production |

**vLLM memory configuration**

```python
from vllm import LLM

# vLLM pre-allocates the entire KV cache at startup
# gpu_memory_utilization controls what fraction of GPU memory is reserved
# Set lower (0.85) if you see OOM; higher (0.95) to maximise KV cache
llm = LLM(
    model='meta-llama/Llama-3-8B-Instruct',
    gpu_memory_utilization=0.90,
    max_model_len=4096,           # limits max context, reduces KV cache
    quantization='awq',           # INT4 weights: halves weight memory
    kv_cache_dtype='fp8',         # FP8 KV cache: halves KV cache memory
)
```

## 5. Exercises

1. **GQA savings**: LLaMA-3 8B uses GQA with n_kv_heads=8 instead of MHA with n_kv_heads=32.
   Compute the exact KV cache memory savings at batch=16, seq=4096. At what batch size
   does GQA allow fitting on a 40 GB GPU where MHA would not?

2. **Safety margin calibration**: Measure `torch.cuda.memory_allocated()` vs
   `torch.cuda.memory_reserved()` during generation at batch sizes 1, 4, 8.
   What is the typical fragmentation overhead? Adjust the safety margin accordingly.

3. **Context window limit**: For the LLaMA-3 8B model on a single A100-80GB,
   find the maximum seq_len at batch_size=1 using the memory solver.
   How does this compare to the model's official max_seq_len of 8192?

4. **FP8 KV cache**: vLLM supports fp8 KV cache (1 byte per element vs 2).
   Recompute the memory budgets in Panel 4 with kv_dtype_bytes=1.
   How much does this increase the maximum supported context at batch=16?

5. **Tensor vs pipeline parallelism**: Implement a simulation where TP requires
   `2 * hidden_size * dtype_bytes * batch_size` bytes of all-reduce communication
   per layer, and PP requires `batch_size * hidden_size * dtype_bytes` bytes of
   point-to-point transfer per pipeline boundary. At what model size and batch size
   does PP become more communication-efficient than TP?

---

## 6. References

- [Rajbhandari et al. (2020) -- ZeRO: Memory Optimizations Toward Training Trillion Parameter Models](https://arxiv.org/abs/1910.02054)
- [Ainslie et al. (2023) -- GQA: Training Generalised Multi-Query Transformer Models from Multi-Head Checkpoints](https://arxiv.org/abs/2305.13245)
- [vLLM Memory Management Documentation](https://docs.vllm.ai/en/latest/serving/engine_args.html)
- [Kwon et al. (2023) -- Efficient Memory Management for LLM Serving with PagedAttention](https://arxiv.org/abs/2309.06180)
